# Гаура-лила: графика на бесплатной видеокарте Google Colab

Этот блокнот делает то, что в облачном контейнере Claude идёт медленно (на процессоре):

1. **Рендер фонов в Blender на GPU** — двор дома Джаганнатхи Мишры (день / вечер / ночь) в 1920×1080 или 4K, плюс карта глубины.
2. **Живописная дорисовка** (по желанию) — Stable Diffusion XL + ControlNet по глубине: картинка становится «как написанная маслом», а геометрия остаётся на месте (важно для игры: невидимые прокси-стены должны совпадать с картинкой).
3. **3D-модель по картинке** (по желанию) — бесплатный Hugging Face Space TRELLIS: из листа персонажа GPT получается черновая 3D-модель.

Всё сохраняется на ваш Google Диск в папку `Gauranga-game/colab-out` — оттуда Claude заберёт файлы.

**Перед запуском:** меню «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU**. Затем «Выполнить всё» (Ctrl+F9). Первый запуск ≈ 10–15 минут на скачивание.

In [ ]:
#@title 1. Диск и видеокарта
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
OUT = '/content/drive/MyDrive/Gauranga-game/colab-out'
os.makedirs(OUT, exist_ok=True)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)

In [ ]:
#@title 2. Blender 4.5, скрипты сцены из репозитория, ассеты Poly Haven (CC0)
BRANCH = 'claude/confident-wozniak-kf6cze'  #@param {type:"string"}
RAW = f'https://raw.githubusercontent.com/sankarshanservant-hub/Gauranga/{BRANCH}/game/probe'
%cd /content
!test -d blender || (wget -q https://download.blender.org/release/Blender4.5/blender-4.5.9-linux-x64.tar.xz && tar xf blender-4.5.9-linux-x64.tar.xz && mv blender-4.5.9-linux-x64 blender && rm blender-4.5.9-linux-x64.tar.xz)
!apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 libgl1 > /dev/null
!wget -q -O build_courtyard.py {RAW}/build_courtyard.py && wget -q -O ph_get.py {RAW}/ph_get.py
!python3 ph_get.py assets tex:reed_roof_04 tex:patterned_clay_plaster tex:clay_floor_001 tex:brown_mud_dry tex:forest_leaves_02 tex:bark_brown_02 tex:bamboo_wall tex:leafy_grass tex:bark_willow tex:mud_forest tex:dirt_floor tex:clay_plaster hdri:qwantani_moonrise hdri:qwantani_mid_morning hdri:qwantani_night mod:tree_small_02 mod:island_tree_01 mod:island_tree_02 mod:shrub_04 mod:pachira_aquatica_01 mod:calathea_orbifolia_01 mod:brass_diya_lantern mod:ceramic_pot mod:planter_pot_clay mod:wicker_basket_01 mod:brass_pot_01 mod:wooden_stool_02 mod:bananas | tail -3
!./blender/blender --version | head -1

In [ ]:
#@title 3. Рендер фонов на GPU (день / вечер / ночь)
RES_PERCENT = 100  #@param {type:"slider", min:25, max:200, step:25}
SAMPLES = 256      #@param {type:"integer"}
PRESETS = {
  'day':   '--hdri qwantani_mid_morning --hdri_rot 120 --sky 1.0 --moon 0 --lamps 0.15 --exposure 0',
  'dusk':  '--hdri qwantani_moonrise --hdri_rot 120 --sky 0.5 --moon 0.8 --exposure 0.3',
  'night': '--hdri qwantani_night --hdri_rot 120 --sky 0.12 --moon 0.6 --lamps 2.5 --exposure 0.6',
}
import time
for name, p in PRESETS.items():
    t = time.time()
    extra = f'--depth {OUT}/depth_{name}.png' if name == 'day' else ''
    !./blender/blender -b -P build_courtyard.py -- --gpu --assets assets --out {OUT}/courtyard_{name}.png --res {RES_PERCENT} --samples {SAMPLES} {p} {extra} 2>&1 | grep -E "GPU|RENDERED|Error"
    print(name, 'готово за', round(time.time() - t), 'с')

In [ ]:
#@title 4. Посмотреть результат
from IPython.display import display, Image as IPImage
for n in ['day', 'dusk', 'night']:
    p = f'{OUT}/courtyard_{n}.png'
    if os.path.exists(p): display(IPImage(p, width=960))

## Живописная дорисовка (SDXL + ControlNet по глубине) — по желанию

Берёт рендер и карту глубины и «переписывает» картинку в живописной манере, сохраняя геометрию.
`STRENGTH` — насколько сильно менять (0.25–0.45: стиль меняется, композиция остаётся). Модели открытые
(SDXL — лицензия CreativeML Open RAIL++-M), скачиваются с Hugging Face (≈ 10 ГБ, первый раз долго).

In [ ]:
#@title 5. Установка diffusers
!pip -q install diffusers==0.31.0 transformers accelerate safetensors

In [ ]:
#@title 6. Дорисовка
import torch, numpy as np
from PIL import Image, ImageOps
from diffusers import ControlNetModel, StableDiffusionXLControlNetImg2ImgPipeline, AutoencoderKL
NAME = 'day'  #@param ["day", "dusk", "night"]
STRENGTH = 0.35  #@param {type:"slider", min:0.1, max:0.7, step:0.05}
PROMPT = "isometric painted background of a 15th-century Bengali village courtyard, thatched mud huts, tulasi altar with rice-paste alpana, banana trees and coconut palms, lush greenery, oil painting, Baldur's Gate style, rich painterly brushwork, soft light, highly detailed"  #@param {type:"string"}
NEG = "photo, blurry, modern objects, text, watermark, people, distorted geometry"
cn = ControlNetModel.from_pretrained('diffusers/controlnet-depth-sdxl-1.0', torch_dtype=torch.float16)
vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=torch.float16)
pipe = StableDiffusionXLControlNetImg2ImgPipeline.from_pretrained('stabilityai/stable-diffusion-xl-base-1.0',
        controlnet=cn, vae=vae, torch_dtype=torch.float16, variant='fp16')
pipe.enable_model_cpu_offload()
src = Image.open(f'{OUT}/courtyard_{NAME}.png').convert('RGB')
depth = Image.open(f'{OUT}/depth_day.png').convert('L')
depth = ImageOps.autocontrast(depth, cutoff=1).convert('RGB')
W, H = 1344, 768   # рабочий размер SDXL (16:9)
img = pipe(PROMPT, negative_prompt=NEG, image=src.resize((W, H), Image.LANCZOS), control_image=depth.resize((W, H)),
           strength=STRENGTH, controlnet_conditioning_scale=0.8, num_inference_steps=30, guidance_scale=6.0).images[0]
img = img.resize(src.size, Image.LANCZOS)
img.save(f'{OUT}/courtyard_{NAME}_painted.png')
display(img.resize((960, 540)))

## 3D-модель персонажа по картинке (TRELLIS, бесплатный Space) — по желанию

Загрузите в папку `Gauranga-game/ref` на Диске лист персонажа (вид спереди, нейтральный фон), например
`mishra_front.png`. Модель получится черновой: Claude затем почистит её в Blender, оденет и сделает риг.
Если Space перегружен — попробуйте позже (бесплатная очередь).

In [ ]:
#@title 7. Картинка → 3D (GLB)
!pip -q install gradio_client
from gradio_client import Client, handle_file
import shutil
REF = '/content/drive/MyDrive/Gauranga-game/ref/mishra_front.png'  #@param {type:"string"}
client = Client('trellis-community/TRELLIS')
client.predict(api_name='/start_session')
img = client.predict(image=handle_file(REF), api_name='/preprocess_image')
res = client.predict(image=handle_file(img), multiimages=[], seed=0, ss_guidance_strength=7.5, ss_sampling_steps=12,
                     slat_guidance_strength=3.0, slat_sampling_steps=12, multiimage_algo='stochastic',
                     mesh_simplify=0.95, texture_size=1024, api_name='/generate_and_extract_glb')
glb = res[2] if isinstance(res[2], str) else res[2].get('value', res[1])
dst = os.path.join(OUT, os.path.splitext(os.path.basename(REF))[0] + '.glb')
shutil.copy(glb, dst)
print('сохранено:', dst)
